# Notebook 12 — Análisis de Class Weights

**Proyecto:** SelvaSonic-ML
**Fase:** S4 extendida — mitigación de desbalance

## Propósito

Calcular y visualizar los **class weights** que serán usados en el próximo entrenamiento del modelo para mitigar el desbalance severo del dataset. Los pesos se calculan usando la fórmula inversamente proporcional:

$$w_c = \frac{N}{K \cdot n_c}$$

donde $N$ es el total de muestras de entrenamiento, $K = 11$ es el número de clases y $n_c$ es el número de muestras de la clase $c$.

**Salida del notebook:** archivo `data/class_weights.pt` con el tensor de pesos listo para usar en `CrossEntropyLoss(weight=class_weights, ...)`.

> ⚠️ **Importante:** los pesos se calculan **únicamente** sobre el split de entrenamiento. Usar val o test introduciría data leakage.

## 1. Setup e imports

In [ ]:
from __future__ import annotations

# Configurar path del proyecto (asumiendo notebook en notebooks/)
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

# Librerías estándar
import numpy as np
import torch
import matplotlib.pyplot as plt

# Módulos del proyecto
from src import config
from src.dataset import create_dataloaders
from src.class_weights import (
    compute_class_weights,
    verify_class_weights,
    extract_labels_from_dataset,
)

# Estilo de matplotlib (consistente con el resto del proyecto)
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["figure.dpi"] = 100  # display
plt.rcParams["savefig.dpi"] = 150  # save

print("✓ Imports OK")

## 2. Cargar el dataset de entrenamiento

Usamos `create_dataloaders()` del proyecto y nos quedamos únicamente con el split de **train**. El argumento `batch_size` no afecta el cálculo de pesos (operamos sobre los labels, no sobre features).

In [ ]:
# Cargar los DataLoaders. Nos interesa el train_dataset, no los loaders en sí.
train_loader, val_loader, test_loader, label_map = create_dataloaders(
    batch_size=32,
    num_workers=0,  # 0 en notebook para evitar problemas en Windows
)

# Acceder al Dataset subyacente (no al DataLoader)
train_dataset = train_loader.dataset

print(f"Train dataset size: {len(train_dataset)}")
print(f"Número de clases: {len(label_map)}")
print(f"Label map: {label_map}")

## 3. Extraer labels del train set

Usamos `extract_labels_from_dataset()` que está optimizada para no cargar los espectrogramas — solo accede a los labels precomputados en el `_index` del Dataset.

In [ ]:
# Extracción eficiente (no carga audios)
labels = extract_labels_from_dataset(train_dataset)

print(f"Total de labels extraídos: {len(labels)}")
print(f"Rango de labels: [{min(labels)}, {max(labels)}]")
print(f"Tipo de elemento: {type(labels[0]).__name__}")

## 4. Calcular class weights

Aplicamos la fórmula $w_c = N / (K \cdot n_c)$ y verificamos las propiedades matemáticas.

In [ ]:
NUM_CLASSES = len(label_map)

result = compute_class_weights(
    labels=labels,
    num_classes=NUM_CLASSES,
)

# Verificación de propiedades matemáticas
is_valid = verify_class_weights(result=result)
print(f"Verificación matemática: {'✓ PASS' if is_valid else '✗ FAIL'}")
print(f"Weighted sum (debe ser ~1.0): {result.weighted_sum:.10f}")
print(f"Tipo del tensor: {result.weights.dtype}")
print(f"Shape del tensor: {result.weights.shape}")

## 5. Tabla de pesos por clase

Esta tabla exportable es la que va al reporte académico.

In [ ]:
# Construir tabla ordenada por número de muestras (descendente)
# Para invertir el label_map: {nombre: idx} -> {idx: nombre}
idx_to_name = {idx: name for name, idx in label_map.items()}

# Ordenamos por cantidad de muestras (más a menos)
sorted_classes = sorted(
    range(NUM_CLASSES),
    key=lambda c: result.class_counts[c],
    reverse=True,
)

print(f"{'#':>3} | {'Especie':<30} | {'Muestras':>9} | {'Freq (%)':>9} | {'Peso':>7}")
print(f"{'-'*3}-+-{'-'*30}-+-{'-'*9}-+-{'-'*9}-+-{'-'*7}")
for rank, cls in enumerate(sorted_classes, start=1):
    name = idx_to_name[cls]
    count = result.class_counts[cls]
    freq = result.class_frequencies[cls] * 100
    weight = result.weights[cls].item()
    print(f"{rank:>3d} | {name:<30} | {count:>9d} | {freq:>8.2f}% | {weight:>7.4f}")

## 6. Visualización 1 — Distribución de muestras vs pesos

Comparación visual directa: a la izquierda el desbalance del dataset, a la derecha cómo los pesos lo compensan.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Nombres cortos para los ticks (las especies largas no caben)
class_indices = list(range(NUM_CLASSES))
class_names_short = [
    idx_to_name[i][:15] + "..." if len(idx_to_name[i]) > 15 else idx_to_name[i]
    for i in class_indices
]

# --- Panel izquierdo: muestras por clase ---
counts = result.class_counts
# Color especial para no_ave (la clase dominante)
no_ave_idx = label_map.get("no_ave", 0)
colors_left = ["#d9534f" if i == no_ave_idx else "#5bc0de" for i in class_indices]

bars_left = axes[0].bar(class_indices, counts, color=colors_left, edgecolor="black", linewidth=0.5)
axes[0].set_xlabel("Clase", fontsize=12)
axes[0].set_ylabel("Número de muestras", fontsize=12)
axes[0].set_title("Distribución del train set: severo desbalance", fontsize=13)
axes[0].set_xticks(class_indices)
axes[0].set_xticklabels(class_names_short, rotation=45, ha="right", fontsize=9)
axes[0].grid(True, alpha=0.3, linestyle="--", axis="y")

# Anotar el valor sobre cada barra
for bar, count in zip(bars_left, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.01,
                 f"{count}", ha="center", va="bottom", fontsize=8)

# --- Panel derecho: pesos por clase ---
weights = result.weights.numpy()
colors_right = ["#d9534f" if i == no_ave_idx else "#5cb85c" for i in class_indices]

bars_right = axes[1].bar(class_indices, weights, color=colors_right, edgecolor="black", linewidth=0.5)
axes[1].axhline(y=1.0, color="gray", linestyle="--", alpha=0.6, label="Peso = 1.0 (referencia)")
axes[1].set_xlabel("Clase", fontsize=12)
axes[1].set_ylabel("Peso $w_c$", fontsize=12)
axes[1].set_title("Pesos calculados: penalizan errores en clases raras", fontsize=13)
axes[1].set_xticks(class_indices)
axes[1].set_xticklabels(class_names_short, rotation=45, ha="right", fontsize=9)
axes[1].grid(True, alpha=0.3, linestyle="--", axis="y")
axes[1].legend(loc="upper right", fontsize=9)

# Anotar el valor sobre cada barra
for bar, w in zip(bars_right, weights):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(weights)*0.01,
                 f"{w:.2f}", ha="center", va="bottom", fontsize=8)

plt.suptitle("SelvaSonic — Compensación del desbalance mediante class weights",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()

# Guardar para el reporte
output_path = PROJECT_ROOT / "results" / "class_weights_distribution.png"
output_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path, dpi=150, bbox_inches="tight")
print(f"✓ Figura guardada en: {output_path}")
plt.show()

## 7. Visualización 2 — Efecto del peso sobre la pérdida

Esta gráfica muestra **cuántas veces más** pesa un error en cada clase comparado con la clase más común (`no_ave`). Es la métrica que mejor comunica el efecto práctico del weighting.

In [ ]:
# Calcular el ratio: peso_clase / peso_de_no_ave
no_ave_weight = result.weights[no_ave_idx].item()
ratios = (result.weights / no_ave_weight).numpy()

# Ordenar por ratio (descendente)
sorted_idx = np.argsort(ratios)[::-1]
sorted_names = [idx_to_name[i] for i in sorted_idx]
sorted_ratios = ratios[sorted_idx]

fig, ax = plt.subplots(figsize=(12, 7))

# Color gradiente: más rojo = mayor penalización
colors = plt.cm.Reds(np.linspace(0.3, 0.85, NUM_CLASSES))

bars = ax.barh(range(NUM_CLASSES), sorted_ratios, color=colors, edgecolor="black", linewidth=0.5)
ax.set_yticks(range(NUM_CLASSES))
ax.set_yticklabels(sorted_names, fontsize=10)
ax.set_xlabel("Penalización relativa (veces más costoso vs error en no_ave)", fontsize=12)
ax.set_title("Costo relativo de un error por clase\n(con respecto al costo de errar en no_ave)",
             fontsize=13, fontweight="bold")
ax.axvline(x=1.0, color="black", linestyle="--", alpha=0.5, label="Igual costo")
ax.grid(True, alpha=0.3, linestyle="--", axis="x")
ax.invert_yaxis()  # mayor arriba

# Anotar valor en cada barra
for bar, ratio in zip(bars, sorted_ratios):
    ax.text(bar.get_width() + max(sorted_ratios)*0.01, bar.get_y() + bar.get_height()/2,
            f"{ratio:.1f}x", va="center", fontsize=10, fontweight="bold")

ax.legend(loc="lower right", fontsize=10)
plt.tight_layout()

output_path = PROJECT_ROOT / "results" / "class_weights_ratios.png"
plt.savefig(output_path, dpi=150, bbox_inches="tight")
print(f"✓ Figura guardada en: {output_path}")
plt.show()

## 8. Guardar el tensor de pesos para el entrenamiento

Persistimos el tensor en `data/class_weights.pt` para que el notebook de entrenamiento lo cargue directamente sin tener que recalcular.

> **Nota:** este archivo es ligero (~50 bytes para 11 floats), pero igual no lo incluimos en git porque puede regenerarse desde el dataset. Lo agregamos a `.gitignore`.

In [ ]:
# Guardar el tensor
weights_path = PROJECT_ROOT / "data" / "class_weights.pt"
weights_path.parent.mkdir(parents=True, exist_ok=True)

# Guardamos también metadata útil para reproducibilidad
save_payload = {
    "weights": result.weights,
    "class_counts": torch.from_numpy(result.class_counts),
    "class_frequencies": torch.from_numpy(result.class_frequencies),
    "num_classes": NUM_CLASSES,
    "total_samples": int(result.class_counts.sum()),
    "label_map": label_map,
    "weighted_sum_check": result.weighted_sum,
}
torch.save(save_payload, weights_path)

print(f"✓ Pesos guardados en: {weights_path}")
print(f"  Tamaño del archivo: {weights_path.stat().st_size} bytes")

# Smoke test: cargar y verificar
loaded = torch.load(weights_path, weights_only=False)
assert torch.allclose(loaded["weights"], result.weights), "Error en guardado/carga"
print(f"✓ Smoke test de carga OK")
print(f"  Pesos cargados: {loaded['weights']}")

## 9. Conclusiones y siguiente paso

### Hallazgos clave

1. **Desbalance confirmado:** la clase `no_ave` domina con más de 50% del dataset, mientras que las clases raras tienen menos del 3% cada una.
2. **Penalización calculada:** los pesos van desde ~0.17 (no_ave) hasta ~3-6 para las clases más raras. Esto implica que un error en una clase rara va a contribuir hasta **~30-40 veces más** a la pérdida que un error en `no_ave`.
3. **Verificación matemática:** la propiedad de normalización $\sum_c (n_c/N) w_c = 1$ se cumple, lo que garantiza que la escala de la pérdida ponderada es comparable con la pérdida sin pesos.

### Lo que esto debería lograr en el entrenamiento

- **F1-score por clase más uniforme:** las clases raras deberían dejar de tener F1 ≈ 0.
- **Accuracy global posiblemente menor en val/test:** es un trade-off esperado y deseable. El modelo dejará de "hacer trampa" prediciendo no_ave por defecto.
- **Confusion matrix más diagonal:** menos confusión sistemática entre clases similares (especialmente intra-género).

### Próximo paso

Usar `data/class_weights.pt` en el training notebook V5 junto con `label_smoothing=0.1` en `CrossEntropyLoss`.